In [ ]:
#| default_exp features.mds_exon

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
import re
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class MDSExonEvaluator(FeatureEvaluator):
    """Exon-level MDS distributions."""
    
    name = "MDSExon"
    source_file = ".MDS.exon.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "gene" in cols and "mds" in cols:
                gene_grouped = df.groupby("gene")["mds"]
                for gene, vals in gene_grouped:
                    if str(gene) == "NAN": continue
                    g = str(gene).replace(" ", "_").replace("-","_")
                    extracted[f"{g}_mds_exon_mean"] = float(vals.mean())
                    if len(vals) > 1:
                        extracted[f"{g}_mds_exon_std"] = float(vals.std())
    
            return extracted
        except Exception as e:
            log.warning("mds_exon_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = MDSExonEvaluator()
df = pd.DataFrame([{"gene": "TP53", "mds": 0.8}, {"gene": "TP53", "mds": 0.4}])
res = evaluator.extract(df)
assert "TP53_mds_exon_mean" in res
assert "TP53_mds_exon_std" in res
